# Laboratorio de regresión - 5

|                |   |
:----------------|---|
| **Nombre**     | Cecilia Quiroz Hidalgo  |
| **Fecha**      | 20/09/2026  |
| **Expediente** | 757942  |

## Validación

Hemos estado usando `train_test_split` en nuestros modelos anteriores.

¿Por qué?

- Para separar una parte de los datos que el modelo no ve mientras entrena, y usarla después para probarlo. Así podemos medir qué tan bien predice con datos nuevos. Si probáramos el modelo con los mismos datos con los que entrenó, el error saldría demasiado bueno y no detectaríamos problemas como el overfitting.

Si la muestra es un subset de la población y queremos generalizar sobre la población, ¿no sería mejor utilizar todos los datos al entrenar un modelo?

- Si conviene usar todos los datos, pero solo después de validar. Si entrenamos con todo desde el principio, ya no queda ningún dato nuevo para probar el modelo, y no sabríamos si realmente generaliza o si solo memorizó los datos.

El propósito de volver a muestrear dentro de nuestro dataset es tener una idea de qué tan buena podría ser la generalización de nuestro modelo. Imagina un dataset ya separado en dos mitades. Utilizas la primera mitad para entrenar el modelo y pruebas en la segunda mitad; la segunda mitad eran datos invisibles para el modelo al momento de entrenar. Esto nos lleva a tres escenario típicos:

1. Si el modelo hace buenas predicciones en la segunda mitad, significa que la primera mitad era "suficiente" para generalizar.
2. Si el modelo no hace buenas predicciones en la segunda mitad, pero sí en la primera mitad, podría ser que había información importante en la segunda mitad que debió haber sido tomada en cuenta al entrenar, o un problema de overfitting.
3. Si el modelo no hace buenas predicciones en la segunda mitad, y tampoco en la primera mitad, se tendrían que revisar los factores y/o el modelo seleccionado.

El caso ideal sería el 1, pero por estadística los errores y varianzas tienen como entrada el número de muestas, por lo que tenemos menos seguridad de nuestros resutados al usar menos muestras. Si vemos que el modelo generaliza bien podemos unir de nuevo el dataset y entrenar sobre el dataset completo.

En el caso 2 está el problema de que no podemos saber qué información es necesaria para el entrenamiento apropiado del modelo; esto nos lleva a pensar que debemos usar el dataset completo para entrenar, pero esto nos lleva al mismo problema de no saber si el modelo puede generalizar.

El problema sólo incrementa si se tienen hiperparámetros en el modelo (e.g. $\lambda$ en regularización).

## Leave-One-Out Cross Validation

Este método de validación es una colección de $n$ `train-test-split`. Teniendo un dataset de $n$ muestras, la lógica es:
1. Saca una muestra del dataset.
2. Entrena tu modelo con las $n-1$ muestras.
3. Evalúa tu modelo en la muestra que quedó fuera con el métrico que más se ajuste a la aplicación.
4. Regresa la muestra al dataset.
5. Repite 1-4 con muestras diferentes hasta haber hecho el procedimiento $n$ veces para $n$ muestras.
6. Calcula la media y desviación estándar de los métricos guardados.

Con los resultados del proceso de validación podemos saber qué tan bueno podría ser el modelo seleccionado con los datos (con/sin transformaciones).

### Ejercicio 1

Utiliza el dataset `Motor Trend Car Road Tests`. Elimina la columna `model` y entrena 32 modelos diferentes utilizando Leave-One-Out Cross Validation con target `mpg`. Utiliza MSE como métrico.

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.linear_model import LinearRegression

In [5]:
df = pd.read_excel("Motor Trend Car Road Tests.xlsx")
df = df.drop(columns="model")

In [6]:
df.head()

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [7]:
df.columns

Index(['mpg', 'cyl', 'disp', 'hp', 'drat', 'wt', 'qsec', 'vs', 'am', 'gear',
       'carb'],
      dtype='object')

In [9]:
X = df[['cyl', 'disp', 'hp', 'drat', 'wt', 'qsec', 'vs', 'am', 'gear',
       'carb']]
y = df['mpg']

In [11]:
scores = cross_val_score(LinearRegression(), X, y,
                         cv=LeaveOneOut(),
                         scoring="neg_mean_squared_error")
mse_lista = -scores

In [13]:
print("Modelos entrenados:", len(mse_lista))
print("MSE promedio:", np.mean(mse_lista))
print("Desviación estándar del MSE:", np.std(mse_lista))
print("Desviación estándar de mpg:", y.std())

Modelos entrenados: 32
MSE promedio: 12.181558006901948
Desviación estándar del MSE: 17.06739987188853
Desviación estándar de mpg: 6.026948052089104


Interpreta.

- Se hicieron 32 modelos, usando 31 autos para entrenar y probando cada modelo con el auto que quedó fuera.
- El MSE promedio fue de ~12.18. Como la varianza de mpg es ~36, el modelo predice mejor que simplemente usar el promedio.
- La desviación estándar del MSE es alta, lo que significa que el error cambia mucho entre los autos. Esto puede deberse a que solo hay 32 autos.

Conclusión: el modelo funciona de manera aceptable, pero no es muy estable.

## K-Folds Cross-Validation

El dataset `Motor Trend Car Road Tests` sólo tiene 32 muestras, y utilizar un modelo sencillo de regresión múltiple hace que usar LOOCV sea muy rápido. El dataset `California Housing` tiene $20640$ muestras para $9$ columnas, entonces realizar un ajuste sobre una transformación o sobre el modelo y luego calcular el impacto esperado podría tomar más tiempo.

La solución propuesta es dividir el dataset en *k* folds (partes iguales), ajustar en *k-1* folds y probar en el restante.

### Ejercicio 2
Utiliza el dataset `California Housing` y haz K-folds Cross Validation con 10 folds. Utiliza el MSE como métrico.

In [14]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print("Dataset Shape:", housing.data.shape, housing.target.shape)
print("Dataset Features:", housing.feature_names)
print("Dataset Target:", housing.target_names)
X = housing.data
y = housing.target

Dataset Shape: (20640, 8) (20640,)
Dataset Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Dataset Target: ['MedHouseVal']


In [16]:
from sklearn.model_selection import KFold

In [17]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(LinearRegression(), X, y,
                         cv=kf,
                         scoring="neg_mean_squared_error")
mse_lista = -scores

In [18]:
print("MSE de cada fold:", np.round(mse_lista, 3))
print("MSE promedio:", np.mean(mse_lista))
print("Desviación estándar del MSE:", np.std(mse_lista))
print("Desviación estándar del target:", y.std())

MSE de cada fold: [0.559 0.553 0.481 0.574 0.489 0.528 0.551 0.475 0.564 0.55 ]
MSE promedio: 0.5325266619260597
Desviación estándar del MSE: 0.03522806775487835
Desviación estándar del target: 1.153928204041225


Interpreta.

- El MSE promedio fue de ~0.53. Como la varianza del objetivo es ~1.33, el modelo tiene un error menor que simplemente usar el promedio.
- La desviación estándar del MSE es muy pequeña, por lo que los resultados de los 10 grupos son muy parecidos.
 
Conclusión: la estimación es estable y confiable, aunque el modelo todavía tiene un error considerable.

## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3